### ADC Setup

     gcloud init

1. choose option 1
2. if your GCP account is visible, choose the option or else  go dor sign in with a new google account
3. choose project: bdc-trainings
4. default compute region and zone: Y
5. region: option 10 -> uscentral1-b



        gcloud auth application-default login


# Prompt Management with Vertex AI SDK

------------------------------------
Detailed Jupyter notebook tutorial on managing prompts using Vertex AI
Includes: creating, versioning, listing, modifying, inference, deletion, and tagging


## Setup


1. initializing a new prompt and registering to vertex AI prompt registry
2. Fetch/Retrieve an existing prompt from Vertex AI prompt registry - make inference with the help of retrieved prompt
3. Create a new version of an existing prompt by modifying that prompt
4. List all the prompts from a specific project 
5. list all versions of a sepcific prompt
6. Delete the specific prompt on vertex ai prompt registry

In [2]:
!pip install --upgrade google-cloud-aiplatform --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.30.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.


In [1]:
import os
from google.cloud import aiplatform
from vertexai import Client, types
from google import genai
from google.genai import types as genai_types


In [2]:
PROJECT = "bdc-trainings"
LOCATION = "us-central1"
CLIENT = Client(project=PROJECT, location=LOCATION)
print(f"Vertex AI client created for project {PROJECT}, location {LOCATION}")


Vertex AI client created for project bdc-trainings, location us-central1


## Register the prompt to Vertex AI



In [4]:
## Define a local prompt

define_prompt = types.Prompt(
    prompt_data=types.PromptData(
        contents=[
            genai_types.Content(parts=[genai_types.Part(text="Hello, {name}! How are you doing today?")])
        ],
        variables=[{"name": genai_types.Part(text="Alice")}, {"name": genai_types.Part(text="Bob")}],
        model="models/gemini-2.5-flash",
    )
)
print("Local prompt defined.")



Local prompt defined.


In [6]:
registered_prompt = CLIENT.prompts.create(prompt=define_prompt,config={"prompt_display_name": "anshu-sample-prompt-1"})
print("Prompt registered with ID:", registered_prompt.prompt_id)


Prompt registered with ID: 5516689091447488512


## Retrieve a prompt and its version, run inference

In [7]:
retrieved_prompt = CLIENT.prompts.get(prompt_id=registered_prompt.prompt_id)
print("Retrieved prompt ID:", retrieved_prompt.prompt_id)
print(retrieved_prompt.prompt_data)


Retrieved prompt ID: 5516689091447488512
generation_config=None tool_config=None tools=None safety_settings=None contents=[Content(
  parts=[
    Part(
      text='Hello, {name}! How are you doing today?'
    ),
  ]
)] system_instruction=None variables=[{'name': Part(
  text='Alice'
)}, {'name': Part(
  text='Bob'
)}] model='models/gemini-2.5-flash'


In [9]:
# current_version = CLIENT.prompts.get_version(
#     prompt_id=retrieved_prompt.prompt_id, version_id=retrieved_prompt.version_id
# )
# print("Retrieved prompt version:", current_version.version_id)


## Modify and create a new version


In [ ]:
# Step 1: Get your existing prompt by ID
existing_prompt = CLIENT.prompts.get(prompt_id=retrieved_prompt.prompt_id)

# Step 2: Modify it
existing_prompt.prompt_data = types.PromptData(
        contents=[
            genai_types.Content(parts=[genai_types.Part(text="Summarize the provided content into 2 lines {{content}}")])
        ],
        model="gemini-2.5-flash",
    )

# Step 3: Create version — do NOT pass prompt_id, the prompt object already has it
new_version = CLIENT.prompts.create_version(
    prompt=existing_prompt,
    config={'version_display_name':"v2-improved"}
)

print(f"Version ID: {new_version.version_id}")
print(f"Prompt ID: {new_version.prompt_id}")


Version ID: 1
Prompt ID: 8614461947636613120


In [13]:
CLIENT.prompts.create_version?

Signature:
CLIENT.prompts.create_version(
    *,
    prompt: Union[vertexai._genai.types.common.Prompt, vertexai._genai.types.common.PromptDict],
    prompt_id: Optional[str] = None,
    config: Union[vertexai._genai.types.common.CreatePromptVersionConfig, vertexai._genai.types.common.CreatePromptVersionConfigDict, NoneType] = None,
) -> vertexai._genai.types.common.Prompt
Docstring:
Creates a prompt resource and an initial prompt version.

When creating new prompt and prompt version resources, this waits for
the create operation to complete before returning.

Note: This method is recommended instead of create() since it creates a
versioned resource for your prompt.

Args:
  prompt: The prompt to create.
  prompt_id: This parameter is deprecated, since this method will create a new prompt each time it is called. If provided, it will be ignored.
  config: Optional configuration for creating the prompt and prompt version.

Returns:
  A types.Prompt object representing the prompt with its

In [ ]:
modified_prompt = types.Prompt(
    prompt_data=types.PromptData(
        contents=[
            genai_types.Content(parts=[genai_types.Part(text="Hi, {name}! Welcome back. How may I help you today?")])
        ],
        variables=[{"name": genai_types.Part(text="Charlie")}, {"name": genai_types.Part(text="Dana")}],
        model="gemini-2.5-flash",
    )
)
# new_version = CLIENT.prompts.create_version(prompt=modified_prompt, prompt_id=registered_prompt.prompt_id)
# print("New prompt version created:", new_version.version_id)


In [41]:
# List prompts and versions

print("Listing prompts in project:")
for p in CLIENT.prompts.list():
    print(" •", p.prompt_id)

print(f"Listing versions for prompt {registered_prompt.prompt_id}:")
for v in CLIENT.prompts.list_versions(prompt_id=registered_prompt.prompt_id):
    print(" • Version ID:", v.version_id)


Listing prompts in project:
 • 1771383051335499776
 • 1609253464750161920
 • 5529636940376178688
 • 8017875734498377728
 • 7023706116756340736
 • 5608449933855162368
 • 2762174969357008896
 • 8873559663698771968
 • 3827276281230131200
 • 6567716654485078016
 • 3800254683465908224
 • 4241607446948216832
 • 3879067676944891904
 • 7587781970084495360
 • 7632817966358200320
 • 3021131947930812416
 • 5230147565156040704
 • 4501690325428862976
 • 6202925084668067840
 • 4203326850115567616
 • 7700371960768757760
 • 7720638159091924992
 • 942903238829539328
 • 6388881088227311616
 • 4790103220510785536
Listing versions for prompt 1609253464750161920:


In [42]:
#  Run inference with a loaded prompt
genai_client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION)
contents = retrieved_prompt.prompt_data.contents

assembled_prompt = retrieved_prompt.assemble_contents()




response = genai_client.models.generate_content(
    model='gemini-2.5-flash', contents=assembled_prompt
)
print("Generated response:", response.text)


Generated response: Hello Alice and Bob!

As an AI, I don't "do" in the human sense, but I'm ready and available to help with whatever you need. How can I assist you both today?


In [32]:
# # Label or alias a prompt version
# alias_prompt = CLIENT.prompts.create(
#     prompt=types.Prompt(
#         prompt_data=types.PromptData(
#             contents=[
#                 genai_types.Content(parts=[genai_types.Part(text="Production: Hello {name}, how may I assist you today?")])
#             ],
#             variables=[{"name": genai_types.Part(text="Eve")}, {"name": genai_types.Part(text="Frank")}],
#             model="models/gemini-2.5-flash",
#         ),
#     ),
#     config=types.CreatePromptConfig(
#         display_name="my-prompt-prod-anshu",
#         labels={"environment": "production"},
#     ),
# )
# print("Alias version created:", alias_prompt.version_id, alias_prompt.labels)


## Delete a prompt version and entire prompt


In [33]:
# CLIENT.prompts.delete_version(
#     prompt_id=registered_prompt.prompt_id, version_id=new_version.version_id
# )
# print(f"Deleted version {new_version.version_id}")

CLIENT.prompts.delete(prompt_id=registered_prompt.prompt_id)
print(f"Deleted prompt {registered_prompt.prompt_id}")


Deleted prompt 6547450456161910784


In [ ]:
# # Restore a previous version
# restored = CLIENT.prompts.restore_version(
#     prompt_id=registered_prompt.prompt_id, version_id="your-version-id-to-restore"
# )
# print("Restored version ID:", restored.version_id)

